In [4]:
import pandas as pd
import numpy as np

import tensorflow.keras as keras
from keras import Sequential
from keras.layers import *
from keras.optimizers import Adam
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

# Dataset

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/es_en.csv")
df.head()

,en,es
0,I hope you're not alone.,Espero que no estés solo.
1,"When I was taking a bath, the telephone rang.","Mientras me bañaba, sonó el teléfono."
2,I just need you to come with me.,Solo necesito que vengas conmigo.
3,Tom wondered how soon Mary would have dinner r...,Tom se preguntaba cuán pronto María tendría li...
4,Tom is waiting for an answer.,Tom está esperando una respuesta.


In [6]:
df.shape

(10000, 2)

In [7]:
# spanish
es_sentences = df.es.values
es_tokenizer = Tokenizer()
es_tokenizer.fit_on_texts(es_sentences)
es_sequences = es_tokenizer.texts_to_sequences(es_sentences)

In [10]:
# english
en_sentences = df.en.values
en_tokenizer = Tokenizer()
en_tokenizer.fit_on_texts(en_sentences)
en_sequences = en_tokenizer.texts_to_sequences(en_sentences)

In [19]:
# longitud y vocabulario

es_max_length = max([len(ss) for ss in es_sequences])
en_max_length = max([len(ss) for ss in en_sequences])

es_vocab = len(es_tokenizer.word_index) + 1
en_vocab = len(en_tokenizer.word_index) + 1

In [22]:
# padding

es_sequences_padded = pad_sequences(es_sequences, maxlen=es_max_length, truncating='post')
en_sequences_padded = pad_sequences(en_sequences, maxlen=en_max_length, truncating='post')

# Model

In [25]:
keras.utils.set_random_seed(812)

embed_dim = 128
lstm_units = 64

model = Sequential([
    Embedding(
        input_dim=es_vocab,
        output_dim=embed_dim,
        input_length=es_max_length
    ),
    LSTM(lstm_units, return_sequences=False),
    RepeatVector(en_max_length),
    LSTM(lstm_units, return_sequences=True, dropout=.2),
    TimeDistributed(Dense(en_vocab, activation='softmax'))
])

In [26]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(1e-3),
    metrics=['accuracy']
)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 31, 128)           1010304   
                                                                 
 lstm (LSTM)                 (None, 64)                49408     
                                                                 
 repeat_vector (RepeatVecto  (None, 25, 64)            0         
 r)                                                              
                                                                 
 lstm_1 (LSTM)               (None, 25, 64)            33024     
                                                                 
 time_distributed (TimeDist  (None, 25, 5053)          328445    
 ributed)                                                        
                                                                 
Total params: 1421181 (5.42 MB)
Trainable params: 142118

In [27]:
%%time

model.fit(
    es_sequences_padded, en_sequences_padded,
    epochs=35
)

Epoch 1/35


313/313 [==============================] - 26s 72ms/step - loss: 2.8938 - accuracy: 0.7462
Epoch 2/35
313/313 [==============================] - 23s 73ms/step - loss: 2.1815 - accuracy: 0.7502
Epoch 3/35
313/313 [==============================] - 23s 72ms/step - loss: 2.1802 - accuracy: 0.7502
Epoch 4/35
313/313 [==============================] - 22s 72ms/step - loss: 2.1798 - accuracy: 0.7502
Epoch 5/35
313/313 [==============================] - 22s 71ms/step - loss: 2.1795 - accuracy: 0.7502
Epoch 6/35
313/313 [==============================] - 22s 71ms/step - loss: 2.1792 - accuracy: 0.7502
Epoch 7/35
313/313 [==============================] - 23s 72ms/step - loss: 2.1790 - accuracy: 0.7502
Epoch 8/35
313/313 [==============================] - 23s 72ms/step - loss: 2.1784 - accuracy: 0.7502
Epoch 9/35
313/313 [==============================] - 22s 72ms/step - loss: 2.1783 - accuracy: 0.7502
Epoch 10/35
313/313 [==============================] - 23s 72ms/step - loss: 2.1

In [54]:
# model.save("model_seq2seq_35.h5")

In [63]:
from keras.models import load_model

model2 = load_model("model_seqseq2_900.h5")

# Predicciones

In [69]:
ii = 852

print(f"{es_sentences[ii]}")
print(f"{en_sentences[ii]}")

preds = model2.predict(es_sequences_padded[ii:ii+1])[0]
' '.join([en_tokenizer.index_word[ww] for ww in np.argmax(preds, 1) if ww != 0])

Nosotros habremos vivido aquí durante diez años para el término de este mes.
We will have lived here for ten years at the end of this month.
1/1 [==============================] - 0s 25ms/step


'we can have should long here ten at at at the of this month'